# Manual llama.cpp CUDA Setup (Windows)

Run this notebook one time to prepare `llama.cpp` + `llama-server` for GPU usage before running the app.

This notebook does not start the FastAPI app. It only prepares tools and optionally starts `llama-server` for testing.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

import requests
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()

ROOT = Path.cwd().resolve()
LLAMA_CPP_DIR = (ROOT / "llama.cpp").resolve()
BUILD_DIR = LLAMA_CPP_DIR / "build"
GGUF_MODELS_DIR = (ROOT / "models" / "gguf").resolve()
GGUF_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Default HF model requested by user
HF_GGUF_REPO_ID = os.getenv("HF_GGUF_REPO_ID", "H4miid/qwen2_5_coder_7b_merged_f16.gguf").strip()
HF_GGUF_FILENAME = os.getenv("HF_GGUF_FILENAME", "").strip()
if not HF_GGUF_FILENAME:
    HF_GGUF_FILENAME = HF_GGUF_REPO_ID.rsplit("/", 1)[-1]

HF_GGUF_REVISION = os.getenv("HF_GGUF_REVISION", "main").strip()
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

print(f"ROOT={ROOT}")
print(f"LLAMA_CPP_DIR={LLAMA_CPP_DIR}")
print(f"GGUF_MODELS_DIR={GGUF_MODELS_DIR}")
print(f"HF default model={HF_GGUF_REPO_ID}")

In [ ]:
# Check required tools first
required_tools = ["git", "cmake"]
missing = [tool for tool in required_tools if shutil.which(tool) is None]
if missing:
    raise RuntimeError(f"Missing required tools on PATH: {missing}. Install them first.")
print("All required tools found on PATH.")

In [ ]:
# Clone llama.cpp if missing
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp ...")
    subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], check=True)
else:
    print("llama.cpp already exists. Skipping clone.")

In [ ]:
# Configure and build with CUDA
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print("Configuring CMake (CUDA ON) ...")
subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"], cwd=str(BUILD_DIR), check=True)

print("Building llama.cpp ...")
subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j"], cwd=str(BUILD_DIR), check=True)

print("Build completed.")

In [ ]:
# Locate llama-server executable
candidates = [
    BUILD_DIR / "bin" / "Release" / "llama-server.exe",
    BUILD_DIR / "bin" / "llama-server.exe",
    BUILD_DIR / "bin" / "llama-server",
]
llama_server_exe = next((p for p in candidates if p.exists()), None)

if llama_server_exe is None:
    hits = [f for f in BUILD_DIR.rglob("llama-server*") if f.is_file()]
    llama_server_exe = hits[0] if hits else None

if llama_server_exe is None:
    raise FileNotFoundError("llama-server executable not found after build.")

print(f"llama-server: {llama_server_exe}")

## Optional: Resolve GGUF model and verify llama-server

This section supports both options:
- `LOCAL_GGUF_PATH` in `.env` (direct local file path)
- HF download fallback (default: `H4miid/qwen2_5_coder_7b_merged_f16.gguf`)

Optional HF auth in `.env`:
- `HF_TOKEN` or `HUGGINGFACEHUB_API_TOKEN`

In [ ]:
# Resolve GGUF model path (local first, then HF download fallback)
def resolve_gguf_model_path() -> Path:
    local_path = os.getenv("LOCAL_GGUF_PATH", "").strip()
    if local_path:
        p = Path(local_path).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"LOCAL_GGUF_PATH does not exist: {p}")
        print(f"Using LOCAL_GGUF_PATH: {p}")
        return p

    target = GGUF_MODELS_DIR / HF_GGUF_FILENAME
    if target.exists():
        print(f"Using cached HF model: {target}")
        return target

    print(f"Downloading HF model: {HF_GGUF_REPO_ID} / {HF_GGUF_FILENAME}")
    downloaded = hf_hub_download(
        repo_id=HF_GGUF_REPO_ID,
        filename=HF_GGUF_FILENAME,
        revision=HF_GGUF_REVISION,
        token=HF_TOKEN,
        local_dir=str(GGUF_MODELS_DIR),
        local_dir_use_symlinks=False,
    )
    p = Path(downloaded).resolve()
    print(f"Downloaded GGUF: {p}")
    return p

GGUF_MODEL_PATH = resolve_gguf_model_path()
print(f"Resolved GGUF_MODEL_PATH={GGUF_MODEL_PATH}")

In [ ]:
# Optional runtime check
HOST = os.getenv("LLAMA_SERVER_HOST", "127.0.0.1").strip()
PORT = int(os.getenv("LLAMA_SERVER_PORT", "8081"))

if not str(GGUF_MODEL_PATH).strip():
    print("GGUF_MODEL_PATH is empty. Resolve model path first.")
else:
    cmd = [
        str(llama_server_exe),
        "--model",
        str(GGUF_MODEL_PATH),
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--ctx-size",
        "8192",
        "--n-gpu-layers",
        "999",
        "--threads",
        str(max(1, (os.cpu_count() or 4) - 1)),
    ]
    print("Starting llama-server for health check ...")
    print(" ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    health_ok = False
    for _ in range(90):
        time.sleep(1)
        try:
            r = requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
            if r.status_code == 200:
                health_ok = True
                break
        except Exception:
            pass

    print("Health check:", "OK" if health_ok else "FAILED")
    if health_ok:
        print("Set these in .env for app runtime:")
        print(f"LLAMA_SERVER_URL=http://{HOST}:{PORT}/v1")
        print("LLAMA_OPENAI_MODEL=local-gguf")
        print(f"LOCAL_GGUF_PATH={GGUF_MODEL_PATH}")

    # Stop test process. Comment out if you want to keep it running.
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except Exception:
        proc.kill()